In [ ]:
import requests
import torch

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

val = ds['validation'].to_pandas()

val.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379 entries, 0 to 378
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    379 non-null    object
 1   label   379 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 6.1+ KB


In [3]:
val['label'] = val['label'].apply(lambda x: 'business' if x == 0 else 'entertainment' if x == 1 else 'politics' if x == 2 else 'sport' if x == 3 else 'tech')

labels = val['label'].unique()

val

,text,label
0,Why Cell will get the hard sell The world is c...,tech
1,Iranian MPs threaten mobile deal Turkey's bigg...,business
2,Google launches TV search service The net sear...,tech
3,"Dutch bank to lay off 2,850 staff ABN Amro, th...",business
4,Franz Ferdinand's art school lesson Scottish r...,entertainment
...,...,...
374,Karachi stocks hit historic high The Karachi S...,business
375,Dance music not dead says Fatboy DJ Norman Coo...,entertainment
376,Real in talks over Gravesen move Real Madrid a...,sport
377,Man Utd stroll to Cup win Wayne Rooney made a ...,sport


In [4]:
def generate_text(message, url):
    payload = {
        "model": "deepseek-r1:1.5b",
        "prompt": message,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_predict": 5000,
        },
        "raw": True
    }

    response = requests.post(url, json=payload)

    return response.json()

In [5]:
def classify(text, labels):

    url = "http://localhost:11434/api/generate"

    message = f"""[INST]
        System: You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions.
        User: Classify the following text based on the task: Category classification of news articles. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}
        [/INST]
        """

    response = generate_text(message, url)

    done_reason = response['done_reason']
    num_tokens = response['eval_count']
    response = response['response'].strip().lower()


    if done_reason != 'length':
        content = response.split('</think>')[1]
    else:
        return "loop", num_tokens, response

    if 'business' in content:
        content = 'business'
    elif 'sport' in content:
        content = 'sport'
    elif 'entertainment' in content:
        content = 'entertainment'
    elif 'politics' in content:
        content = 'politics'
    elif 'tech' in content:
        content = 'tech'
    else:
        content = 'error'

    print(content, num_tokens)

    return content, num_tokens, response

In [ ]:
val['prediction'], val['num_tokens'], val['response'] = zip(*val['text'].apply(lambda x: classify(x, labels)))

business 435
business 532
entertainment 958
tech 18
tech 55
politics 562
business 740
sport 551
entertainment 247
tech 1411
entertainment 389
entertainment 472
sport 396
politics 688
entertainment 118
error 458
entertainment 1331
politics 535
sport 515
sport 754
business 640
politics 42
sport 262
business 617
entertainment 410
entertainment 570
entertainment 608
business 525
entertainment 562
politics 733
entertainment 353
business 842
tech 19
sport 413
politics 918
tech 21
tech 115
politics 64
tech 22
business 294
sport 95
tech 93
sport 957
politics 659
sport 332
business 477
entertainment 923
politics 30
sport 738
business 429
tech 23
politics 18
business 405
tech 323
entertainment 991
business 481
sport 511
business 483
business 996
entertainment 295
business 730
business 915
business 295
politics 601
business 671
politics 82
tech 57
entertainment 542
tech 2075
business 679
sport 297
politics 66
politics 351
tech 21
entertainment 438
sport 505
business 428
business 619
tech 889
tech

In [7]:
val

,text,label,prediction,num_tokens,response
0,Why Cell will get the hard sell The world is c...,tech,business,435,[text] [labels]\n\nto determine whether you ca...
1,Iranian MPs threaten mobile deal Turkey's bigg...,business,business,532,so i have to classify this text into one of th...
2,Google launches TV search service The net sear...,tech,entertainment,958,"[text]\n [options: tech, business, ent..."
3,"Dutch bank to lay off 2,850 staff ABN Amro, th...",business,tech,18,[inst] \n answer with the correct labe...
4,Franz Ferdinand's art school lesson Scottish r...,entertainment,tech,55,[inst]\n answer with the correct label...
...,...,...,...,...,...
374,Karachi stocks hit historic high The Karachi S...,business,business,115,[inst]\n text: the text is about the p...
375,Dance music not dead says Fatboy DJ Norman Coo...,entertainment,business,115,"[inst]\n please reason step by step, a..."
376,Real in talks over Gravesen move Real Madrid a...,sport,sport,555,"-- so, i need to figure out which category thi..."
377,Man Utd stroll to Cup win Wayne Rooney made a ...,sport,sport,120,[inst]\n text: the text is about the f...


In [8]:
val.prediction.value_counts()

prediction
business         110
tech              69
politics          68
sport             55
entertainment     51
loop              20
error              6
Name: count, dtype: int64

In [12]:
max_num_tokens = val[val['prediction'] != 'loop']['num_tokens'].max()
print(f'Max number of tokens: {max_num_tokens}')

Max number of tokens: 3043


In [ ]:
average_num_tokens = val[(val['prediction'] != 'loop') & (val['prediction'] != 'error')]['num_tokens'].mean()
print(f'Average number of tokens: {average_num_tokens}')

Average number of tokens: 424.5269121813031


In [9]:
val.to_csv('reasoning_size_val.csv', index=False)